In [3]:
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# EVALUATE SEASONAL IDEAL UI/API CASE-STUDY PREDICTIONS
# ============================================================
#
# Expected folder structure:
#
# C:/IDEAL_Programming/New_Evaluation/SEASONAL/
#   summer_home100_20170809/
#       actual_target_day.csv
#       case_metadata.json
#       ui_pred_combined_no_history_....csv
#       ui_pred_combined_with_history_....csv
#
# The script:
#   - finds prediction files automatically by filename substring
#   - evaluates RF / XGB / LGBM / ensemble from combined CSVs
#   - saves metrics and plots per seasonal case
#   - saves global seasonal summaries
#
# ============================================================

# ============================================================
# CONFIG
# ============================================================

ROOT_DIR = Path(r"C:/IDEAL_Programming/New_Evaluation/SEASONAL")

ACTUAL_FILE = "actual_target_day.csv"
METADATA_FILE = "case_metadata.json"

PREDICTION_FILE_PATTERNS = {
    "no_history": "*ui_pred_combined_no_history*.csv",
    "with_history": "*ui_pred_combined_with_history*.csv",
}

RESULTS_DIR_NAME = "Evaluation_results"

TIME_COL = "timestamp"
ACTUAL_COL = "actual_consumption_Wh"

SAVE_ENCODING = "utf-8-sig"

DAY_NAMES_GR = {
    0: "Δευτέρα",
    1: "Τρίτη",
    2: "Τετάρτη",
    3: "Πέμπτη",
    4: "Παρασκευή",
    5: "Σάββατο",
    6: "Κυριακή",
}


# ============================================================
# HELPERS
# ============================================================

def find_case_dirs(root: Path):
    case_dirs = []

    for p in sorted(root.iterdir()):
        if p.is_dir() and (p / METADATA_FILE).exists():
            case_dirs.append(p)

    return case_dirs


def load_metadata(case_dir: Path):
    path = case_dir / METADATA_FILE

    if not path.exists():
        return {}

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def find_latest_file(case_dir: Path, pattern: str):
    matches = sorted(case_dir.glob(pattern))

    if not matches:
        return None, []

    latest = max(matches, key=lambda p: p.stat().st_mtime)
    return latest, matches


def maybe_convert_to_wh(series, col_name):
    values = pd.to_numeric(series, errors="coerce")
    name = str(col_name).lower()

    if "kwh" in name:
        values = values * 1000.0

    return values


def load_actual(actual_path: Path):
    df = pd.read_csv(actual_path, low_memory=False)

    if TIME_COL not in df.columns:
        raise ValueError(f"Missing timestamp column in {actual_path}")

    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")

    actual_col_candidates = [
        "consumption_Wh",
        "actual_consumption_Wh",
        "actual_Wh",
        "actual",
        "y_true",
        "target",
    ]

    actual_col = None
    for col in actual_col_candidates:
        if col in df.columns:
            actual_col = col
            break

    if actual_col is None:
        raise ValueError(
            f"Could not detect actual consumption column in {actual_path}. "
            f"Columns: {list(df.columns)}"
        )

    out = df[[TIME_COL, actual_col]].copy()
    out = out.rename(columns={actual_col: ACTUAL_COL})
    out[ACTUAL_COL] = maybe_convert_to_wh(out[ACTUAL_COL], actual_col)

    out = (
        out
        .dropna(subset=[TIME_COL, ACTUAL_COL])
        .groupby(TIME_COL, as_index=False)
        .agg({ACTUAL_COL: "mean"})
        .sort_values(TIME_COL)
        .reset_index(drop=True)
    )

    return out


def detect_combined_prediction_columns(df: pd.DataFrame):
    """
    Expected combined CSV columns:
      rf_Wh, xgb_Wh, lgbm_Wh, ensemble_Wh

    Also supports:
      pred_consumption_Wh
    """
    preferred = ["rf_Wh", "xgb_Wh", "lgbm_Wh", "ensemble_Wh"]
    found = [c for c in preferred if c in df.columns]

    if found:
        return found

    fallback = []

    for col in df.columns:
        c = str(col).lower()

        if col == TIME_COL:
            continue

        if "weight" in c:
            continue

        if c in [
            "remaining_kwh_today",
            "total_kwh_day",
            "cutoff_iso",
            "external_temp_c",
            "is_weekend",
            "is_holiday",
        ]:
            continue

        if c.endswith("_wh") or c in ["pred_consumption_wh", "prediction_wh"]:
            numeric = pd.to_numeric(df[col], errors="coerce")
            if numeric.notna().sum() > 0:
                fallback.append(col)

    if fallback:
        return fallback

    raise ValueError(
        "Could not detect prediction columns. "
        f"Available columns: {list(df.columns)}"
    )


def rename_prediction_col(col: str, scenario: str):
    c = str(col).lower()

    if c == "rf_wh" or c.endswith("_rf"):
        return f"{scenario}_rf"

    if c == "xgb_wh" or c.endswith("_xgb"):
        return f"{scenario}_xgb"

    if c == "lgbm_wh" or c.endswith("_lgbm"):
        return f"{scenario}_lgbm"

    if c == "ensemble_wh" or "ensemble" in c:
        return f"{scenario}_ensemble"

    if c == "pred_consumption_wh":
        return f"{scenario}_prediction"

    clean = (
        str(col)
        .replace("_Wh", "")
        .replace("_wh", "")
        .replace("pred_", "")
        .replace("prediction_", "")
    )

    return f"{scenario}_{clean}"


def load_prediction_file(pred_path: Path, scenario: str, actual_timestamps):
    df = pd.read_csv(pred_path, low_memory=False)

    if TIME_COL in df.columns:
        df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    else:
        if len(df) == len(actual_timestamps):
            df[TIME_COL] = list(pd.to_datetime(actual_timestamps))
        else:
            raise ValueError(
                f"Missing timestamp column and cannot align by row count: {pred_path}"
            )

    pred_cols = detect_combined_prediction_columns(df)

    out = df[[TIME_COL] + pred_cols].copy()

    rename_map = {}

    for col in pred_cols:
        new_col = rename_prediction_col(col, scenario)
        rename_map[col] = new_col
        out[col] = maybe_convert_to_wh(out[col], col)

    out = out.rename(columns=rename_map)

    out = (
        out
        .groupby(TIME_COL, as_index=False)
        .mean(numeric_only=True)
        .sort_values(TIME_COL)
        .reset_index(drop=True)
    )

    prediction_labels = list(rename_map.values())

    preferred_order = [
        f"{scenario}_rf",
        f"{scenario}_xgb",
        f"{scenario}_lgbm",
        f"{scenario}_ensemble",
        f"{scenario}_prediction",
    ]

    prediction_labels = (
        [c for c in preferred_order if c in prediction_labels]
        + [c for c in prediction_labels if c not in preferred_order]
    )

    return out, prediction_labels


def mape_safe(y_true, y_pred, eps=1e-6):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mask = y_true > eps
    if mask.sum() == 0:
        return np.nan

    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100.0)


def smape_safe(y_true, y_pred, eps=1e-6):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    denom = np.abs(y_true) + np.abs(y_pred) + eps
    return float(np.mean(2.0 * np.abs(y_pred - y_true) / denom) * 100.0)


def compute_metrics(df: pd.DataFrame, pred_col: str):
    tmp = df[[TIME_COL, ACTUAL_COL, pred_col]].dropna().copy()

    if tmp.empty:
        return None

    y_true = tmp[ACTUAL_COL].astype(float).values
    y_pred = tmp[pred_col].astype(float).values

    actual_total_kWh = float(y_true.sum() / 1000.0)
    pred_total_kWh = float(y_pred.sum() / 1000.0)
    daily_abs_error_kWh = abs(pred_total_kWh - actual_total_kWh)

    actual_peak_idx = int(np.argmax(y_true))
    pred_peak_idx = int(np.argmax(y_pred))

    actual_peak_timestamp = pd.Timestamp(tmp.iloc[actual_peak_idx][TIME_COL])
    pred_peak_timestamp = pd.Timestamp(tmp.iloc[pred_peak_idx][TIME_COL])

    actual_peak_hour = int(actual_peak_timestamp.hour)
    pred_peak_hour = int(pred_peak_timestamp.hour)

    return {
        "rows": int(len(tmp)),
        "MAE_Wh": float(np.mean(np.abs(y_true - y_pred))),
        "RMSE_Wh": float(math.sqrt(np.mean((y_true - y_pred) ** 2))),
        "MAPE_%": mape_safe(y_true, y_pred),
        "sMAPE_%": smape_safe(y_true, y_pred),
        "bias_Wh": float(np.mean(y_pred - y_true)),

        "actual_total_kWh": actual_total_kWh,
        "pred_total_kWh": pred_total_kWh,
        "daily_abs_error_kWh": daily_abs_error_kWh,
        "daily_error_pct": (
            float(daily_abs_error_kWh / actual_total_kWh * 100.0)
            if actual_total_kWh > 0
            else np.nan
        ),

        "actual_peak_hour": actual_peak_hour,
        "pred_peak_hour": pred_peak_hour,
        "peak_hour_abs_error": abs(pred_peak_hour - actual_peak_hour),
        "actual_peak_Wh": float(np.max(y_true)),
        "pred_peak_Wh": float(np.max(y_pred)),
        "peak_value_abs_error_Wh": float(abs(np.max(y_pred) - np.max(y_true))),
    }


def scenario_from_col(col):
    if str(col).startswith("no_history"):
        return "no_history"

    if str(col).startswith("with_history"):
        return "with_history"

    return "unknown"


def plot_comparison(df: pd.DataFrame, pred_cols, out_path: Path, title: str):
    plot_df = df.sort_values(TIME_COL).copy()
    plot_df["hour"] = plot_df[TIME_COL].dt.hour

    plt.figure(figsize=(13, 6))

    plt.plot(
        plot_df["hour"],
        plot_df[ACTUAL_COL],
        marker="o",
        linewidth=2.6,
        label="Actual",
    )

    for col in pred_cols:
        if col not in plot_df.columns:
            continue

        plt.plot(
            plot_df["hour"],
            plot_df[col],
            marker="o",
            linewidth=1.8,
            label=col,
        )

    plt.title(title)
    plt.xlabel("Hour of day")
    plt.ylabel("Consumption (Wh)")
    plt.xticks(range(24))
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=8, ncol=2)
    plt.tight_layout()
    plt.savefig(out_path, dpi=180, bbox_inches="tight")
    plt.close()


# ============================================================
# MAIN
# ============================================================

case_dirs = find_case_dirs(ROOT_DIR)

if not case_dirs:
    raise ValueError(f"No case folders found under: {ROOT_DIR}")

print("=" * 140)
print("SEASONAL IDEAL UI/API CASE-STUDY EVALUATION")
print("=" * 140)
print("Root:", ROOT_DIR)
print("Cases:", [p.name for p in case_dirs])
print()

all_metrics = []
all_hourly = []
missing_cases = []

for case_dir in case_dirs:
    metadata = load_metadata(case_dir)

    season = metadata.get("season", "")
    home_id = metadata.get("home_id", "")
    target_date = metadata.get("target_date", "")

    case_label = f"{season}_{home_id}_{target_date}".strip("_")

    results_dir = case_dir / RESULTS_DIR_NAME
    results_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 140)
    print(f"Processing {case_dir.name}")
    print("=" * 140)

    actual_path = case_dir / ACTUAL_FILE

    if not actual_path.exists():
        print(f"[SKIP] Missing actual file: {actual_path}")
        missing_cases.append({"case_folder": case_dir.name, "reason": "missing_actual"})
        continue

    try:
        actual = load_actual(actual_path)
    except Exception as exc:
        print(f"[SKIP] Could not load actual file: {exc}")
        missing_cases.append({"case_folder": case_dir.name, "reason": f"actual_error: {exc}"})
        continue

    comparison = actual.copy()
    actual_timestamps = comparison[TIME_COL].tolist()

    prediction_cols_all = []
    prediction_cols_by_scenario = {}
    selected_prediction_files = {}

    for scenario, pattern in PREDICTION_FILE_PATTERNS.items():
        pred_path, matches = find_latest_file(case_dir, pattern)

        if pred_path is None:
            print(f"[WARN] Missing {scenario} prediction file. Pattern: {pattern}")
            selected_prediction_files[scenario] = None
            continue

        if len(matches) > 1:
            print(f"[INFO] Multiple files found for {scenario}; using latest:")
            for m in matches:
                print("  -", m.name)
            print("  selected:", pred_path.name)

        try:
            pred_df, pred_labels = load_prediction_file(
                pred_path=pred_path,
                scenario=scenario,
                actual_timestamps=actual_timestamps,
            )
        except Exception as exc:
            print(f"[WARN] Could not load {scenario} prediction file: {exc}")
            selected_prediction_files[scenario] = str(pred_path)
            continue

        comparison = comparison.merge(pred_df, on=TIME_COL, how="left")

        prediction_cols_all.extend(pred_labels)
        prediction_cols_by_scenario[scenario] = pred_labels
        selected_prediction_files[scenario] = str(pred_path)

        print(f"{scenario}: loaded {len(pred_df)} rows | file: {pred_path.name}")
        print(f"columns: {pred_labels}")

    if not prediction_cols_all:
        print(f"[SKIP] No prediction columns found for {case_dir.name}")
        missing_cases.append({"case_folder": case_dir.name, "reason": "no_prediction_columns"})
        continue

    # Add context columns
    comparison["season"] = season
    comparison["case_folder"] = case_dir.name
    comparison["home_id"] = home_id
    comparison["target_date"] = target_date
    comparison["date"] = comparison[TIME_COL].dt.date
    comparison["hour"] = comparison[TIME_COL].dt.hour
    comparison["day_of_week"] = comparison[TIME_COL].dt.weekday
    comparison["day_name_gr"] = comparison["day_of_week"].map(DAY_NAMES_GR)
    comparison["is_weekend"] = (comparison["day_of_week"] >= 5).astype(int)

    comparison["history_stability_score"] = metadata.get("history_stability_score")
    comparison["history_stability_category"] = metadata.get("history_stability_category")
    comparison["recommended_max_alpha"] = metadata.get("recommended_max_alpha")
    comparison["recommended_max_alpha_range"] = metadata.get("recommended_max_alpha_range")

    comparison["hometype"] = metadata.get("hometype")
    comparison["residents"] = metadata.get("residents")
    comparison["urban_rural_class"] = metadata.get("urban_rural_class")
    comparison["total_floor_area_m2"] = metadata.get("total_floor_area_m2")
    comparison["num_electric_appliances"] = metadata.get("num_electric_appliances")

    # Save hourly comparison
    preferred_cols = [
        TIME_COL,
        "season",
        "case_folder",
        "home_id",
        "target_date",
        "date",
        "hour",
        "day_name_gr",
        "is_weekend",
        "hometype",
        "residents",
        "urban_rural_class",
        "total_floor_area_m2",
        "num_electric_appliances",
        "history_stability_score",
        "history_stability_category",
        "recommended_max_alpha",
        "recommended_max_alpha_range",
        ACTUAL_COL,
    ] + prediction_cols_all

    preferred_cols = [c for c in preferred_cols if c in comparison.columns]

    hourly_path = results_dir / "hourly_comparison_all_models.csv"
    comparison[preferred_cols].to_csv(hourly_path, index=False, encoding=SAVE_ENCODING)

    # Scenario-specific hourly CSVs
    for scenario, cols in prediction_cols_by_scenario.items():
        scenario_cols = [
            TIME_COL,
            "season",
            "case_folder",
            "home_id",
            "target_date",
            "date",
            "hour",
            "day_name_gr",
            "is_weekend",
            ACTUAL_COL,
        ] + cols

        scenario_cols = [c for c in scenario_cols if c in comparison.columns]

        scenario_path = results_dir / f"hourly_comparison_{scenario}.csv"
        comparison[scenario_cols].to_csv(scenario_path, index=False, encoding=SAVE_ENCODING)

    # Compute metrics
    metric_rows = []

    for pred_col in prediction_cols_all:
        metrics = compute_metrics(comparison, pred_col)

        if metrics is None:
            continue

        row = {
            "season": season,
            "case_folder": case_dir.name,
            "home_id": home_id,
            "target_date": target_date,
            "scenario": scenario_from_col(pred_col),
            "prediction_column": pred_col,

            "hometype": metadata.get("hometype"),
            "residents": metadata.get("residents"),
            "urban_rural_class": metadata.get("urban_rural_class"),
            "total_floor_area_m2": metadata.get("total_floor_area_m2"),
            "num_electric_appliances": metadata.get("num_electric_appliances"),

            "history_stability_score": metadata.get("history_stability_score"),
            "history_stability_category": metadata.get("history_stability_category"),
            "recommended_max_alpha": metadata.get("recommended_max_alpha"),
            "recommended_max_alpha_range": metadata.get("recommended_max_alpha_range"),

            **metrics,
        }

        metric_rows.append(row)
        all_metrics.append(row)

    metrics_df = pd.DataFrame(metric_rows)

    metrics_path = results_dir / "evaluation_metrics.csv"
    metrics_df.to_csv(metrics_path, index=False, encoding=SAVE_ENCODING)

    # Plots
    plot_comparison(
        comparison,
        prediction_cols_all,
        results_dir / "hourly_comparison_all_models.png",
        f"{case_dir.name} - Actual vs all predictions",
    )

    for scenario, cols in prediction_cols_by_scenario.items():
        plot_comparison(
            comparison,
            cols,
            results_dir / f"hourly_comparison_{scenario}.png",
            f"{case_dir.name} - Actual vs {scenario} predictions",
        )

    # Metadata
    eval_metadata = {
        "season": season,
        "case_folder": case_dir.name,
        "home_id": home_id,
        "target_date": target_date,
        "actual_file": str(actual_path),
        "selected_prediction_files": selected_prediction_files,
        "results_dir": str(results_dir),
        "hourly_comparison_file": str(hourly_path),
        "metrics_file": str(metrics_path),
        "prediction_columns_used": prediction_cols_all,
        "prediction_columns_by_scenario": prediction_cols_by_scenario,
        "note": (
            "Prediction files were detected automatically by filename substrings "
            "'ui_pred_combined_no_history' and 'ui_pred_combined_with_history'. "
            "If multiple matching files existed, the most recently modified file was used."
        ),
    }

    with open(results_dir / "evaluation_metadata.json", "w", encoding="utf-8") as f:
        json.dump(eval_metadata, f, ensure_ascii=False, indent=2)

    print()
    print("Metrics:")
    print(
        metrics_df[
            [
                "scenario",
                "prediction_column",
                "rows",
                "MAE_Wh",
                "RMSE_Wh",
                "MAPE_%",
                "sMAPE_%",
                "actual_total_kWh",
                "pred_total_kWh",
                "daily_abs_error_kWh",
                "daily_error_pct",
            ]
        ].to_string(index=False)
    )

    print()
    print("Saved:")
    print("Hourly comparison:", hourly_path)
    print("Metrics:", metrics_path)
    print("Plots:", results_dir)
    print()

    all_hourly.append(comparison[preferred_cols].copy())


# ============================================================
# GLOBAL OUTPUTS
# ============================================================

global_results_dir = ROOT_DIR / RESULTS_DIR_NAME
global_results_dir.mkdir(parents=True, exist_ok=True)

if all_metrics:
    global_metrics = pd.DataFrame(all_metrics)

    global_metrics_path = global_results_dir / "all_seasonal_cases_evaluation_metrics.csv"
    global_metrics.to_csv(global_metrics_path, index=False, encoding=SAVE_ENCODING)

    summary = (
        global_metrics
        .groupby(["scenario", "prediction_column"], as_index=False)
        .agg(
            cases=("case_folder", "nunique"),
            seasons=("season", "nunique"),
            homes=("home_id", "nunique"),

            mean_MAE_Wh=("MAE_Wh", "mean"),
            mean_RMSE_Wh=("RMSE_Wh", "mean"),
            mean_MAPE_pct=("MAPE_%", "mean"),
            mean_sMAPE_pct=("sMAPE_%", "mean"),
            mean_bias_Wh=("bias_Wh", "mean"),
            mean_daily_abs_error_kWh=("daily_abs_error_kWh", "mean"),
            mean_daily_error_pct=("daily_error_pct", "mean"),
            mean_peak_value_abs_error_Wh=("peak_value_abs_error_Wh", "mean"),
        )
        .sort_values(["mean_RMSE_Wh", "mean_MAE_Wh"])
        .reset_index(drop=True)
    )

    summary_path = global_results_dir / "summary_by_scenario_and_model.csv"
    summary.to_csv(summary_path, index=False, encoding=SAVE_ENCODING)

    # Best model per seasonal case
    best_per_case = (
        global_metrics
        .sort_values(["case_folder", "RMSE_Wh", "MAE_Wh"])
        .groupby("case_folder", as_index=False)
        .head(1)
        .reset_index(drop=True)
    )

    best_per_case_path = global_results_dir / "best_model_per_seasonal_case.csv"
    best_per_case.to_csv(best_per_case_path, index=False, encoding=SAVE_ENCODING)

    # Best by season
    best_per_season = (
        global_metrics
        .sort_values(["season", "RMSE_Wh", "MAE_Wh"])
        .groupby("season", as_index=False)
        .head(1)
        .reset_index(drop=True)
    )

    best_per_season_path = global_results_dir / "best_model_per_season.csv"
    best_per_season.to_csv(best_per_season_path, index=False, encoding=SAVE_ENCODING)

    print("=" * 140)
    print("GLOBAL SEASONAL SUMMARY BY SCENARIO / MODEL")
    print("=" * 140)
    print(summary.to_string(index=False))

    print()
    print("=" * 140)
    print("BEST MODEL PER SEASONAL CASE")
    print("=" * 140)

    best_cols = [
        "season",
        "case_folder",
        "home_id",
        "target_date",
        "history_stability_category",
        "recommended_max_alpha",
        "scenario",
        "prediction_column",
        "MAE_Wh",
        "RMSE_Wh",
        "MAPE_%",
        "sMAPE_%",
        "daily_abs_error_kWh",
        "daily_error_pct",
    ]

    best_cols = [c for c in best_cols if c in best_per_case.columns]
    print(best_per_case[best_cols].to_string(index=False))

    print()
    print("Saved global metrics:", global_metrics_path)
    print("Saved global summary:", summary_path)
    print("Saved best per case:", best_per_case_path)
    print("Saved best per season:", best_per_season_path)

else:
    print("[WARN] No metrics computed.")

if all_hourly:
    global_hourly = pd.concat(all_hourly, ignore_index=True)

    global_hourly_path = global_results_dir / "all_seasonal_cases_hourly_comparison.csv"
    global_hourly.to_csv(global_hourly_path, index=False, encoding=SAVE_ENCODING)

    print("Saved global hourly comparison:", global_hourly_path)

if missing_cases:
    missing_df = pd.DataFrame(missing_cases)
    missing_path = global_results_dir / "missing_or_skipped_cases.csv"
    missing_df.to_csv(missing_path, index=False, encoding=SAVE_ENCODING)
    print("Saved missing/skipped cases:", missing_path)

print("=" * 140)
print("DONE")
print("=" * 140)

SEASONAL IDEAL UI/API CASE-STUDY EVALUATION
Root: C:\IDEAL_Programming\New_Evaluation\SEASONAL
Cases: ['autumn_home72_20170927', 'spring_home316_20180423', 'summer_home100_20170809', 'winter_home268_20180111']

Processing autumn_home72_20170927
no_history: loaded 24 rows | file: ui_pred_combined_no_history_Edinburgh_2017-09-27_20260527_131726.csv
columns: ['no_history_rf', 'no_history_xgb', 'no_history_lgbm', 'no_history_ensemble']
with_history: loaded 24 rows | file: ui_pred_combined_with_history_Edinburgh_2017-09-27_20260527_132005.csv
columns: ['with_history_rf', 'with_history_xgb', 'with_history_lgbm', 'with_history_ensemble']

Metrics:
    scenario     prediction_column  rows    MAE_Wh   RMSE_Wh     MAPE_%   sMAPE_%  actual_total_kWh  pred_total_kWh  daily_abs_error_kWh  daily_error_pct
  no_history         no_history_rf    24 15.336638 18.692485  26.641834 22.338429             1.402        1.765114             0.363114        25.899703
  no_history        no_history_xgb    24 91